Installing

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn
!pip install pandas
!pip install imbalanced-learn
!pip install tqdm
!pip install scikit-image
!pip install scipy
!pip install ace_tools


Libraries

In [ ]:
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from skimage import exposure
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
from collections import Counter
import glob
import os

Train

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import normalize

# === CONFIGURATION ===
BASE_PATH = r"/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs"
IMAGE_SIZE = (60, 120)
PATCH_SIZE = 10
stride = 10
NUM_IMAGES = 5
FINGER_LIST = ['L_Fore', 'L_Middle', 'L_Ring', 'R_Fore', 'R_Middle', 'R_Ring']
NUM_COMBINED_FINGERS = 6
LBP_CONFIGS = [(1, 8), (1, 16), (2, 8)]
NUM_ROW_COMPONENTS = 47
NUM_COL_COMPONENTS = 47

# === RIU2 MAPPING ===
def get_riu2_mapping(P):
    table = np.zeros(2 ** P, dtype=np.uint8)
    for i in range(2 ** P):
        binary = [(i >> j) & 1 for j in range(P)]
        rotations = [binary[n:] + binary[:n] for n in range(P)]
        min_rotation = min(rotations)
        extended = min_rotation + [min_rotation[0]]
        transitions = sum(extended[j] != extended[j + 1] for j in range(P))
        table[i] = sum(min_rotation) if transitions <= 2 else P + 1
    return table

mapping_dict = {P: get_riu2_mapping(P) for _, P in LBP_CONFIGS}

# === LBP FEATURE EXTRACTOR ===
def extract_lbp_histogram(block, P, R, riu2_map):
    lbp = local_binary_pattern(block, P, R, method='ror').astype(np.uint16)
    lbp_mapped = riu2_map[lbp]
    hist, _ = np.histogram(lbp_mapped.ravel(), bins=np.arange(0, P + 3), density=True)
    return hist

# === Collect All Fused Finger LBP Images for (2D)^2PCA ===
lbp_images = []
lbp_labels = []

print("\n🔄 Extracting fused RIU2-LBP feature matrices for (2D)^2PCA using Strategy 1 (Protocol 1)...")
print(f"📁 Base Path: {BASE_PATH}")

subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="Subjects"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for img_idx in range(1, NUM_IMAGES + 1):
        combined_matrix = []
        for finger in FINGER_LIST[:NUM_COMBINED_FINGERS]:
            img_path = os.path.join(subject_path, finger, f"{img_idx:02d}.bmp")
            print(f"📥 Reading image from: {img_path}")
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"❌ Could not read image: {img_path}")
                continue

            img = cv2.resize(img, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
            img = cv2.fastNlMeansDenoising(img, h=10)
            img = cv2.equalizeHist(img)

            h_patches = (img.shape[0] - PATCH_SIZE) // stride + 1
            w_patches = (img.shape[1] - PATCH_SIZE) // stride + 1

            lbp_matrix = np.zeros((h_patches, w_patches * sum(P + 2 for _, P in LBP_CONFIGS)))

            for i, y in enumerate(range(0, img.shape[0] - PATCH_SIZE + 1, stride)):
                row_features = []
                for x in range(0, img.shape[1] - PATCH_SIZE + 1, stride):
                    block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                    if block.shape != (PATCH_SIZE, PATCH_SIZE):
                        continue
                    block_hist = []
                    for R, P in LBP_CONFIGS:
                        hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                        block_hist.extend(hist)
                    row_features.append(block_hist)
                if row_features:
                    lbp_matrix[i, :] = np.hstack(row_features)

            combined_matrix.append(lbp_matrix)

        if combined_matrix:
            fused_lbp = np.hstack(combined_matrix)
            lbp_images.append(fused_lbp)
            lbp_labels.append(f"{subj}_img{img_idx:02d}")

# === Compute (2D)^2PCA Projection ===
def compute_2d2pca_projection(images, num_row_components, num_col_components):
    print("\n⚙️ Computing (2D)^2PCA projection matrices...")
    n = len(images)
    h, w = images[0].shape
    mean_img = sum(images) / n
    G_row = np.zeros((h, h))
    G_col = np.zeros((w, w))
    for A in images:
        A = A - mean_img
        G_row += A @ A.T
        G_col += A.T @ A
    G_row /= n
    G_col /= n
    eig_vals_r, eig_vecs_r = np.linalg.eigh(G_row)
    eig_vals_c, eig_vecs_c = np.linalg.eigh(G_col)
    U = eig_vecs_r[:, np.argsort(-eig_vals_r)[:num_row_components]]
    V = eig_vecs_c[:, np.argsort(-eig_vals_c)[:num_col_components]]
    return U, V

U, V = compute_2d2pca_projection(lbp_images, NUM_ROW_COMPONENTS, NUM_COL_COMPONENTS)

# === Project Each Fused LBP Image ===
projected_features = [U.T @ A @ V for A in lbp_images]
flat_features = np.array([f.flatten() for f in projected_features])
lbp_labels = np.array(lbp_labels)

flat_features.shape, lbp_labels[:5]


Test

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern

# === CONFIGURATION ===
BASE_PATH = r"/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs"
IMAGE_SIZE = (60, 120)  # (height, width)
PATCH_SIZE = 10
stride = 10
TEST_INDICES = [6, 7, 8, 9, 10]
FINGER_LIST = ['L_Fore', 'L_Middle', 'L_Ring', 'R_Fore', 'R_Middle', 'R_Ring']
NUM_COMBINED_FINGERS = 6
LBP_CONFIGS = [(1, 8), (1, 16), (2, 8)]

# === RIU2 MAPPING === (should match training)
def get_riu2_mapping(P):
    table = np.zeros(2 ** P, dtype=np.uint8)
    for i in range(2 ** P):
        binary = [(i >> j) & 1 for j in range(P)]
        rotations = [binary[n:] + binary[:n] for n in range(P)]
        min_rotation = min(rotations)
        extended = min_rotation + [min_rotation[0]]
        transitions = sum(extended[j] != extended[j + 1] for j in range(P))
        table[i] = sum(min_rotation) if transitions <= 2 else P + 1
    return table

mapping_dict = {P: get_riu2_mapping(P) for _, P in LBP_CONFIGS}

# === LBP FEATURE EXTRACTOR ===
def extract_lbp_histogram(block, P, R, riu2_map):
    lbp = local_binary_pattern(block, P, R, method='ror').astype(np.uint16)
    lbp_mapped = riu2_map[lbp]
    hist, _ = np.histogram(lbp_mapped.ravel(), bins=np.arange(0, P + 3), density=True)
    return hist

# === EXTRACT TEST FUSED LBP IMAGES ===
test_lbp_images = []
test_labels = []

print(f"\n🧪 Extracting test LBP feature matrices for Strategy 1 (4-finger fusion) - Protocol 1:")
print(f"📁 Test Source Directory: {BASE_PATH}")

subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="Subjects"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for img_idx in TEST_INDICES:
        combined_matrix = []

        for finger in FINGER_LIST[:NUM_COMBINED_FINGERS]:
            img_path = os.path.join(subject_path, finger, f"{img_idx:02d}.bmp")
            print(f"📥 Reading: {img_path}")
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                print(f"❌ Cannot read image: {img_path}")
                continue

            img = cv2.resize(img, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
            img = cv2.fastNlMeansDenoising(img, h=10)
            img = cv2.equalizeHist(img)

            h_patches = (img.shape[0] - PATCH_SIZE) // stride + 1
            w_patches = (img.shape[1] - PATCH_SIZE) // stride + 1

            lbp_matrix = np.zeros((h_patches, w_patches * sum(P + 2 for _, P in LBP_CONFIGS)))

            for i, y in enumerate(range(0, img.shape[0] - PATCH_SIZE + 1, stride)):
                row_features = []
                for x in range(0, img.shape[1] - PATCH_SIZE + 1, stride):
                    block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                    if block.shape != (PATCH_SIZE, PATCH_SIZE):
                        continue
                    block_hist = []
                    for R, P in LBP_CONFIGS:
                        hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                        block_hist.extend(hist)
                    row_features.append(block_hist)
                if row_features:
                    lbp_matrix[i, :] = np.hstack(row_features)

            combined_matrix.append(lbp_matrix)

        if len(combined_matrix) == NUM_COMBINED_FINGERS:
            fused_lbp = np.hstack(combined_matrix)
            test_lbp_images.append(fused_lbp)
            test_labels.append(f"{subj}_img{img_idx:02d}")
            print(f"✅ Features extracted for: {subj}_img{img_idx:02d}")

# === PROJECT TEST IMAGES USING (2D)^2PCA ===
try:
    test_projected = [U.T @ A @ V for A in test_lbp_images]
    test_flat_features = np.array([f.flatten() for f in test_projected])
    test_labels = np.array(test_labels)

    print("\n✅ (2D)²PCA projection complete for test set.")
    print("📐 Projected shape:", test_flat_features.shape)
    print("🧾 Labels preview:", test_labels[:5])

except NameError:
    print("\n❌ Error: Ensure `U` and `V` (projection matrices from training) are available in this script.")


Benchmarking

In [ ]:
# === CLASSIFICATION USING MANHATTAN DISTANCE ===
print("\n🔍 Starting classification using Manhattan distance...")

correct_matches = 0
total_tests = len(test_flat_features)

for i in range(total_tests):
    test_vec = test_flat_features[i]
    true_label = test_labels[i]

    # Compute Manhattan distance to all training vectors
    distances = np.sum(np.abs(flat_features - test_vec), axis=1)
    nearest_index = np.argmin(distances)
    predicted_label = lbp_labels[nearest_index]  # ✅ FIXED

    # Compare subject ID (before "_img")
    true_id = true_label.split('_')[0]
    pred_id = predicted_label.split('_')[0]

    if pred_id == true_id:
        correct_matches += 1
        match = "✅"
    else:
        match = "❌"

    print(f"Test sample {i+1}: Pred = {pred_id}, True = {true_id} {match}")

# === REPORT FINAL ACCURACY ===
accuracy = (correct_matches / total_tests) * 100
print(f"\n🎯 Accuracy = {accuracy:.2f}% ({correct_matches} / {total_tests})")


Session Independent R5

In [ ]:
import numpy as np
from collections import defaultdict

# === Configuration ===
ranks = [1, 5]
rank_correct = defaultdict(int)
total_tests = len(test_labels)

print("📊 Calculating Session-Independent CMC (Rank-1 & Rank-5) — Strategy 1: Fused Fingers...")

for i in range(total_tests):
    proj_test = test_flat_features[i]
    test_label = test_labels[i]

    # ✅ Extract subject only (ignore session for session-independent evaluation)
    test_parts = test_label.split('_')
    test_subject = test_parts[0]
    test_id = test_subject  # No session in ID

    # Compute Manhattan distances
    distances = np.sum(np.abs(flat_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    matched = False
    for r in range(1, max(ranks) + 1):
        candidate_label = lbp_labels[sorted_indices[r - 1]]
        parts = candidate_label.split('_')
        candidate_subject = parts[0]
        candidate_id = candidate_subject  # No session in ID

        if candidate_id == test_id and not matched:
            for k in ranks:
                if r <= k:
                    rank_correct[k] += 1
            matched = True

# === Final CMC Results
for k in ranks:
    accuracy = (rank_correct[k] / total_tests) * 100
    print(f"🎯 Rank-{k} Accuracy (Subject only): {accuracy:.2f}%")


Session Independent CMC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ✅ Helper function: extract subject only (ignore session and finger)
def extract_subject(label):
    return label.split('_')[0]  # subject ID only

# === CONFIGURATION ===
max_rank = 100
rank_correct = np.zeros(max_rank)
total_tests = len(test_flat_features)

print("📊 Calculating Session-Independent CMC Curve (Matching by Subject Only)...")

for i in range(total_tests):
    proj_test = test_flat_features[i]
    true_label = test_labels[i]
    true_subject = extract_subject(true_label)

    # Compute Manhattan distances to all training samples
    distances = np.sum(np.abs(flat_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    # Find the first correct match
    for r in range(max_rank):
        candidate_label = lbp_labels[sorted_indices[r]]
        candidate_subject = extract_subject(candidate_label)

        if candidate_subject == true_subject:
            rank_correct[r:] += 1
            break

# ✅ Normalize to percentage
cmc_curve = (rank_correct / total_tests) * 100

# ✅ Plotting the CMC curve
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, max_rank + 1), cmc_curve, label="Session-Independent CMC", linewidth=2)
plt.xlabel("Rank")
plt.ylabel("Identification Accuracy (%)")
plt.grid(True)
plt.xticks(np.arange(0, max_rank + 1, 10))
plt.legend()
plt.tight_layout()
plt.show()

# ✅ Print key rank accuracies
print(f"🎯 Rank-1 Accuracy   : {cmc_curve[0]:.2f}%")
print(f"🎯 Rank-5 Accuracy   : {cmc_curve[4]:.2f}%")
print(f"🎯 Rank-10 Accuracy  : {cmc_curve[9]:.2f}%")
print(f"🎯 Rank-100 Accuracy : {cmc_curve[99]:.2f}%")


Session Independent Precision, Recall, F1, Accuracy

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# === Initialize lists
all_scores = []
all_labels = []

# === Pairwise score computation for Session-Independent Verification (Strategy 1: Fused Fingers)
for test_idx in range(len(test_flat_features)):
    test_vec = test_flat_features[test_idx]
    test_label = test_labels[test_idx]
    test_parts = test_label.split('_')
    test_subject = test_parts[0]  # ✅ Use only subject
    test_id = test_subject

    for train_idx in range(len(flat_features)):
        train_vec = flat_features[train_idx]
        train_label = lbp_labels[train_idx]
        train_parts = train_label.split('_')
        train_subject = train_parts[0]  # ✅ Use only subject
        train_id = train_subject

        # Skip self-comparison (optional but recommended)
        if test_label == train_label:
            continue

        # Similarity score: negative Manhattan distance
        score = -np.sum(np.abs(test_vec - train_vec))
        all_scores.append(score)

        # Ground truth: genuine if subject matches (session ignored)
        is_genuine = int(test_id == train_id)
        all_labels.append(is_genuine)

# === Normalize similarity scores to [0, 1]
scores = np.array(all_scores)
labels = np.array(all_labels)
scores = (scores - scores.min()) / (scores.max() - scores.min())
# === Toggle SMOTE ===
use_smote = True  # Set True if you want to apply SMO
# === Apply SMOTE (optional)
if use_smote:
    smote = SMOTE(random_state=42)
    scores_2d = scores.reshape(-1, 1)  # Reshape to 2D: (n_samples, 1)
    scores_2d, labels = smote.fit_resample(scores_2d, labels)
    scores = scores_2d.ravel()  # Flatten back to 1D for thresholding
    print("🧪 After SMOTE label distribution:", Counter(labels))
# === Threshold Sweeping to Find Best F1 Score
best_f1 = best_thresh = best_prec = best_rec = 0

for t in np.linspace(0, 1, 1000):
    preds = (scores >= t).astype(int)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        best_prec = precision
        best_rec = recall

# === Final Classification at Optimal Threshold
final_preds = (scores >= best_thresh).astype(int)
accuracy = accuracy_score(labels, final_preds)

# === Print Summary Report
print("🔍 Summary (Session-Independent Verification — Strategy 1: Fused Fingers)")
print("📎 Feature: LBP((8,1),(16,1),(8,2)) + (2D)^2PCA")
print(f"📍 Optimal Threshold  : {best_thresh:.3f}")
print(f"✔️ Accuracy           : {accuracy * 100:.2f}%")
print(f"✔️ Precision (PR)     : {best_prec * 100:.2f}%")
print(f"✔️ Recall (RC)        : {best_rec * 100:.2f}%")
print(f"✔️ F1 Score (F1)      : {best_f1 * 100:.2f}%")
